In [ ]:
from pathlib import Path
import sys
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import optuna

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORTS = PROJECT_ROOT / "reports"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

# Загружаем и строим фичи
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)

folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
    verbose=False,
)

# Загружаем список отобранных фичей
with open(REPORTS / "selected_features_v4.json") as f:
    selected = json.load(f)
features_to_keep = selected["features_to_keep"]

# Готовим X / y
y = train_fe["isFraud"].astype(np.int8)
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "uid"]
X_full = train_fe.drop(columns=drop_cols)
X = X_full[features_to_keep]
cat_cols = [c for c in X.select_dtypes(include=["category"]).columns.tolist()]

print(f"X: {X.shape}")
print(f"Cat cols: {len(cat_cols)}")
print(f"y fraud rate: {y.mean()*100:.2f}%")

In [ ]:
def objective(trial):
    """One Optuna trial: suggest hyperparameters, run full CV, return mean AUC."""
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbose": -1,
        "n_jobs": -1,
        "random_state": 42,

        # Tunable params
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    # Full CV (4 folds из expanding_window)
    splits = expanding_window_splits(folds, n_splits=5)
    fold_scores = []

    for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

        train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
        valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                                  reference=train_data)

        model = lgb.train(
            params=params,
            train_set=train_data,
            num_boost_round=2000,
            valid_sets=[valid_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        preds = model.predict(X_va, num_iteration=model.best_iteration)
        fold_scores.append(roc_auc_score(y_va, preds))

        # Optuna pruning: если после 2 фолдов всё плохо — прерываем trial
        trial.report(np.mean(fold_scores), step=fold_num)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(fold_scores))

print("Objective function defined")

In [ ]:
# Создаём study с TPE sampler + MedianPruner
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)

# Отключаем Optuna логи, чтобы не засорять вывод
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 50

print(f"Starting Optuna study with {N_TRIALS} trials...")
print(f"Baseline (v4): CV AUC 0.92933")
print()

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
elapsed = time.time() - t0

print()
print(f"Completed in {elapsed/60:.1f} min")
print(f"Trials run: {len(study.trials)}")
print(f"Best CV AUC: {study.best_value:.5f}")
print(f"Improvement vs v4: {study.best_value - 0.92933:+.5f}")
print()
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_params = {
    "objective": "binary",
    "metric": "auc",
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
    **study.best_params,
}

# Full CV с лучшими параметрами, сохраняем OOF и модели
splits = expanding_window_splits(folds, n_splits=5)
oof_preds_v5 = np.zeros(len(X), dtype=np.float32)
cv_scores_v5 = []
models_v5 = []

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                              reference=train_data)

    model = lgb.train(
        params=best_params,
        train_set=train_data,
        num_boost_round=3000,   # больше, потому что learning_rate мал
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),  # тоже больше
            lgb.log_evaluation(period=0),
        ],
    )

    oof_preds_v5[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_va, oof_preds_v5[valid_idx])
    cv_scores_v5.append(fold_auc)
    models_v5.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.best_iteration}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores_v5):.5f} ± {np.std(cv_scores_v5):.5f}")
valid_mask = folds > 0
print(f"OOF AUC: {roc_auc_score(y[valid_mask], oof_preds_v5[valid_mask]):.5f}")

print()
print(f"v4 (base LGBM):    CV 0.92933")
print(f"v5 (tuned):        CV {np.mean(cv_scores_v5):.5f} "
      f"(+{np.mean(cv_scores_v5) - 0.92933:+.5f})")

In [ ]:
X_test_full = test_fe.drop(columns=["TransactionID", "TransactionDT", "uid"])
X_test = X_test_full[features_to_keep].copy()

for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(X[col].dtype)

test_preds = np.zeros(len(X_test), dtype=np.float32)
for model in models_v5:
    p = model.predict(X_test, num_iteration=model.best_iteration)
    test_preds += p / len(models_v5)

print(f"Predictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, "
      f"mean={test_preds.mean():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_fe["TransactionID"],
    "isFraud": test_preds,
})
submission.to_csv(DATA_PREDICTIONS / "sub_lgbm_tuned_v5.csv", index=False)

oof_df = pd.DataFrame({
    "TransactionID": train_fe["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds_v5,
    "fold": folds,
})
oof_df.to_csv(DATA_PREDICTIONS / "oof_lgbm_tuned_v5.csv", index=False)

# Сохраним best params для воспроизведения
with open(REPORTS / "best_params_v5.json", "w") as f:
    json.dump({
        "params": study.best_params,
        "cv_auc": float(study.best_value),
        "n_trials": N_TRIALS,
        "notes": "Optuna TPE + MedianPruner, scored on 4-fold expanding-window CV"
    }, f, indent=2)

print("Saved: sub_lgbm_tuned_v5.csv + oof_lgbm_tuned_v5.csv")
print(f"Saved: best_params_v5.json")